In [ ]:
"""
GitHub Trending Parser
Парсит https://github.com/trending и возвращает список трендовых репозиториев.

"""

from dataclasses import dataclass

import requests
from bs4 import BeautifulSoup


@dataclass
class Repository:
    rank: int
    owner: str
    name: str
    full_name: str
    url: str
    description: str
    language: str
    stars_total: int
    forks_total: int
    stars_today: int


def parse_number(text: str) -> int:
    """Converts '1,234' or '1,234 stars today' → int."""
    if not text:
        return 0
    cleaned = text.strip().replace(",", "").split()[0]
    try:
        return int(cleaned)
    except ValueError:
        return 0


def fetch_trending(language: str = "", since: str = "daily") -> list[Repository]:
    """
    Fetches and parses GitHub Trending page.

    Args:
        language: Programming language filter, e.g. 'python', 'typescript'.
                  Empty string means all languages.
        since:    Time range — 'daily', 'weekly', or 'monthly'.

    Returns:
        List of Repository objects.
    """
    base_url = "https://github.com/trending"
    url = f"{base_url}/{language}" if language else base_url
    params = {"since": since}

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    }

    response = requests.get(url, params=params, headers=headers, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.select("article.Box-row")

    repos = []
    for rank, article in enumerate(articles, start=1):
        # Owner / Name
        h2 = article.select_one("h2 a")
        if not h2:
            continue
        parts = [p.strip() for p in h2.get_text(separator="/").split("/") if p.strip()]
        if len(parts) < 2:
            continue
        owner, name = parts[0], parts[1]
        full_name = f"{owner}/{name}"
        repo_url = f"https://github.com/{full_name}"

        # Description
        desc_el = article.select_one("p")
        description = desc_el.get_text(strip=True) if desc_el else ""

        # Language
        lang_el = article.select_one("[itemprop='programmingLanguage']")
        language_val = lang_el.get_text(strip=True) if lang_el else ""

        # Total stars
        stars_link = article.select("a[href$='/stargazers']")
        stars_total = parse_number(stars_link[0].get_text()) if stars_link else 0

        # Total forks
        forks_link = article.select("a[href$='/forks']")
        forks_total = parse_number(forks_link[0].get_text()) if forks_link else 0

        # Stars today
        today_el = article.select_one("span.d-inline-block.float-sm-right")
        stars_today = parse_number(today_el.get_text()) if today_el else 0

        repos.append(
            Repository(
                rank=rank,
                owner=owner,
                name=name,
                full_name=full_name,
                url=repo_url,
                description=description,
                language=language_val,
                stars_total=stars_total,
                forks_total=forks_total,
                stars_today=stars_today,
            )
        )

    return repos


def print_table(repos: list[Repository]) -> None:
    print(f"\n{'#':<4} {'Repository':<45} {'Lang':<14} {'⭐ Total':>10} {'🍴 Forks':>9} {'⭐ Today':>9}")
    print("-" * 95)
    for r in repos:
        print(
            f"{r.rank:<4} {r.full_name:<45} {r.language:<14} "
            f"{r.stars_total:>10,} {r.forks_total:>9,} {r.stars_today:>9,}"
        )
        if r.description:
            print(f"     → {r.description[:90]}")
    print()


repos = fetch_trending(language="", since="daily")
print_table(repos)



#    Repository                                    Lang              ⭐ Total   🍴 Forks   ⭐ Today
-----------------------------------------------------------------------------------------------
1    forrestchang/andrej-karpathy-skills                               9,239       638       702
     → A single CLAUDE.md file to improve Claude Code behavior, derived from Andrej Karpathy's ob
2    TheCraigHewitt/seomachine                     Python              4,717       716       649
     → A specialized Claude Code workspace for creating long-form, SEO-optimized blog content for
3    google-ai-edge/gallery                        Kotlin             19,609     1,832       853
     → A gallery that showcases on-device ML/GenAI use cases and allows people to try and use mod
4    NVIDIA/personaplex                            Python              8,522     1,213       586
     → PersonaPlex code.
5    google-ai-edge/LiteRT-LM                      C++                 3,039       286       501
6 